In [2]:
# Step 1: Install scispacy and the model
#!pip install scispacy
#!pip install spacy
#!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_sm-0.5.3.tar.gz

# Step 2: Import required libraries
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
from collections import Counter

# Step 3: Download and load the model
try:
    nlp = spacy.load("en_core_sci_sm")
    
    # Test the model
    text = "Paracetamol is commonly used to treat fever and mild to moderate pain."
    doc = nlp(text)
    
    # Print results
    print("Named Entities:")
    for ent in doc.ents:
        print(f"Entity: {ent.text}, Label: {ent.label_}")
        
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure you're running this in a fresh Python session")
    print("2. Try restarting your Jupyter kernel")
    print("3. Verify your Python environment has write permissions")

Named Entities:
Entity: Paracetamol, Label: ENTITY
Entity: treat, Label: ENTITY
Entity: fever, Label: ENTITY
Entity: mild, Label: ENTITY
Entity: moderate pain, Label: ENTITY


/opt/anaconda3/envs/NLP/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [3]:
# Multiple ways to get and work with home directory
import os
from pathlib import Path

# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
#print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")

Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi
Current working directory: /Users/naveendeevi/Datasets


In [4]:
df = pd.read_csv("Psych_MOA_V4.csv")

In [5]:
#--------------------------------- Try to RXCUI from medication name ------------------

def get_rxcui_from_name(row):
    """
    Extract RxCUI from the drug name using RxNav API.
    Prioritizes the 'rxcui' field if present. Otherwise,
    fetches using 'medication_name'.
    """
    rxcui = row.get("rxcui")
    drug_name = row.get("medication_name")

    if pd.isna(rxcui) and drug_name:
        try:
            url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name}"
            res = requests.get(url)
            res.raise_for_status()
            data = res.json()
            rxcui = data.get("idGroup", {}).get("rxnormId", [None])[0]
        except Exception as e:
            print(f"[Error] RxCUI lookup failed for '{drug_name}': {e}")
            return None

    if not rxcui:
        print("[Info] No RxCUI or valid drug name available.")
        return None

    return rxcui

#--------------------------------- Mechanism of Action from drug name--------------------------------

def get_chembl_moa(drug_name):
    # Step 1: Search for the ChEMBL ID
    search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    res = requests.get(search_url).json()
    
    try:
        chembl_id = res['molecules'][0]['molecule_chembl_id']
    except (KeyError, IndexError):
        print(f"Drug '{drug_name}' not found.")
        return []

    # Step 2: Get MoA
    moa_url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"
    moa_data = requests.get(moa_url).json()
    mechanisms = moa_data.get("mechanisms", [])

    results = []

    for m in mechanisms:
        target_id = m.get("target_chembl_id")
        target_name = None

        # Fetch target name using target_chembl_id if available
        if target_id:
            target_url = f"https://www.ebi.ac.uk/chembl/api/data/target/{target_id}.json"
            target_res = requests.get(target_url).json()
            target_name = target_res.get("pref_name")

        results.append({
            "mechanism": m.get("mechanism_of_action"),
            "target_chembl_id": target_id,
            "target_name": target_name,
            "action_type": m.get("action_type")
        })

    return results

# Example usage
#print(get_chembl_moa("Biperiden"))

#-----------------------------------Extract mechanism of Action from the drug RXCUI-----------------------------

def get_chembl_moa_from_rxcui(rxcui):
    # Step 1: Get drug name from RxCUI
    
    try:
        rxnav_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{int(rxcui)}/properties.json"
        rxnav_res = requests.get(rxnav_url)
        rxnav_res.raise_for_status()
        drug_name = rxnav_res.json().get("properties", {}).get("name")
    except Exception as e:
        print(f"[Error] Could not resolve drug name from RxCUI {rxcui}: {e}")
        return []

    if not drug_name:
        print(f"[Error] No drug name found for RxCUI {int(rxcui)}")
        return []

    # Step 2: Search for the ChEMBL ID
    try:
        search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
        res = requests.get(search_url).json()
        chembl_id = res['molecules'][0]['molecule_chembl_id']
    except (KeyError, IndexError, Exception) as e:
        print(f"Drug '{drug_name}' not found in ChEMBL: {e}")
        return []

    # Step 3: Get MoA
    moa_url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"
    moa_data = requests.get(moa_url).json()
    mechanisms = moa_data.get("mechanisms", [])

    results = []

    for m in mechanisms:
        target_id = m.get("target_chembl_id")
        target_name = None

        # Fetch target name using target_chembl_id if available
        if target_id:
            target_url = f"https://www.ebi.ac.uk/chembl/api/data/target/{target_id}.json"
            target_res = requests.get(target_url).json()
            target_name = target_res.get("pref_name")

        results.append({
            "mechanism": m.get("mechanism_of_action"),
            "target_chembl_id": target_id,
            "target_name": target_name,
            "action_type": m.get("action_type")
        })

    return results

#----------------------------------- Extract the Mechanism of Action (MOA) of the drug---------------------------
def extract_mechanism(list_x):
    ls = []
    #print(list_x)
    if len(list_x) < 1:
        return None
    else: 
        for i in list_x: 
            ls.append(i['mechanism'])
    #print(ls)
    return ";;; ".join(map(str, ls))

#----------------------------------- Extract the Mechanism of Action (MOA) of the drug---------------------------
def extract_target(list_x):
    ls = []
    if len(list_x) < 1:
        return None
    else: 
        for i in list_x: 
            ls.append(i['target_name'])
    #print(ls)
    return ";;; ".join(map(str, ls))

#------------------------------------ Extract indications of the drug-----------------------------------

def get_chembl_indications_improved(drug_name):
    # Step 1: Get ChEMBL ID
    search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    res = requests.get(search_url).json()
    #print(len(res['molecules']))
    #print('-----------------------')
    print(drug_name)
    try:
        #chembl_id = res["molecules"][0]["molecule_chembl_id"]
        chembl_ids = [m["molecule_chembl_id"] for m in res["molecules"] if "molecule_chembl_id" in m]
        #print(chembl_ids)
    except (KeyError, IndexError):
        #print(f"No ChEMBL ID found for {drug_name}")
        return None
    indications = []
    
    for i in chembl_ids: 
        #print(i)
        ind_url = f"https://www.ebi.ac.uk/chembl/api/data/drug_indication.json?molecule_chembl_id={i}"
        #print(ind_url)
        ind_data = requests.get(ind_url).json()
        #print(type(ind_data))
        try:
            for item in ind_data.get("drug_indications", []):
                #print(item.get("efo_term"))
                indications.append(item.get("efo_term"))
        #print(";;; ".join(map(str, indications)))
                #print(indications)
        except (KeyError, IndexError):
            print(drug_name)
            return None
    indications = list(dict.fromkeys(indications))
    #print(indications)

    if len(indications) > 0: 
        return "; ".join(map(str, indications))
    else: 
        return None
    
'''    
def get_chembl_indications(drug_name):
    # Step 1: Get ChEMBL ID
    search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    res = requests.get(search_url).json()

    try:
        chembl_id = res["molecules"][0]["molecule_chembl_id"]
        #print(chembl_id)
    except (KeyError, IndexError):
        print(f"No ChEMBL ID found for {drug_name}")
        return []

    # Step 2: Get Indications
    ind_url = f"https://www.ebi.ac.uk/chembl/api/data/drug_indication.json?molecule_chembl_id={chembl_id}"
    #print(ind_url)
    ind_data = requests.get(ind_url).json()
    #print(ind_data)
    indications = []
    for item in ind_data.get("drug_indications", []):
        indications.append(item.get("efo_term"))
    #print(";;; ".join(map(str, indications)))
    if len(indications) > 0: 
        return "; ".join(map(str, indications))
    else: 
        return None
'''
#--------------------------------- Get Generic versions of the drug -------------------------------------

def get_ntl_library_med(df,list_of_components):
    #print(type_of_component)
    if list_of_components == 'Clinical Drug Component':
        ls = ["SCDC"]
    elif list_of_components == 'Branded Drug Component':
        ls = ["SBDC"]
    else: 
        ls = ["SCDC","SBDC"]
    """
    Check if drugs in a Pandas DataFrame have generic formulations using RxNorm.

    Parameters:
        df (pd.DataFrame): DataFrame with columns 'rxcui', 'medication_name', 'medication_first_name'

    Returns:
        pd.Series: A Series with generic versions (if any), or empty list
    """
    def get_generic_versions(row,ls):
        #print(ls)
        #print(type(ls))
        #print(type_of_component)
        # Extract RxCUI or fallback to medication_name or medication_first_name
        rxcui = row.get("rxcui")
        drug_name = row.get("medication_name") #or row.get("med_first_name")
        #drug_first_name = row.get("med_first_name")
        #print(rxcui)
        #print(drug_name)

        # Step 1: Get RxCUI if not provided
        if pd.isna(rxcui) and drug_name:
            #print('No rxcui so checking the drug_name')
            try:
                url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name}"
                res = requests.get(url)
                res.raise_for_status()
                data = res.json()
                rxcui = data.get("idGroup", {}).get("rxnormId", [None])[0]
            except Exception as e:
                print(f"[Error] RxCUI lookup failed for '{drug_name}': {e}")
                return []

        #print(rxcui)
        '''
        if pd.isna(rxcui) and drug_first_name:
            #print('Could not get the rxcui from drug name so trying first name')
            try:
                url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_first_name}"
                res = requests.get(url)
                res.raise_for_status()
                data = res.json()
                rxcui = data.get("idGroup", {}).get("rxnormId", [None])[0]
            except Exception as e:
                print(f"[Error] RxCUI lookup failed for '{med_first_name}': {e}")
                return []
        '''
        #print(rxcui)
        
        if not rxcui:
            print("[Info] No RxCUI or drug name available.")
            return []

        # Step 2: Get all related concepts
        try:
            rxcui_v2 = int(rxcui)
            related_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui_v2}/allrelated.json"
            rel_res = requests.get(related_url)
            rel_res.raise_for_status()
            related_data = rel_res.json()
        except Exception as e:
            print(f"[Error] Failed to fetch related concepts for RxCUI {rxcui}: {e}")
            return []

        # Step 3: Extract likely generic drug concepts
        concept_groups = related_data.get("allRelatedGroup", {}).get("conceptGroup", [])
        generic_names = []

        for group in concept_groups:
            #if group.get("tty") in ["SCD", "SCDC", "GPCK"]:
            if group.get("tty") in ls:
                for concept in group.get("conceptProperties", []):
                    name = concept.get("name")
                    if name:
                        generic_names.append(name)

        return generic_names
    return df.apply(lambda x:get_generic_versions(x,ls), axis=1)

# Example usage
#df = pd.DataFrame({
#    'rxcui': ['197361', '5361'],  # Example RxCUI values
#    'medication_name': ['Atomoxetine', 'Gabapentin'],
#    'medication_first_name': ['Lisinopril', '']
#})

#df['generic_versions'] = get_ntl_library_med(df,"Clinical Drug Component")
#print(df)

#-------------------------------------Drug role picked from Chat GPT--------------------------------------------

def drug_role(med_class): 
    augmentation = [
        'Aldehyde Dehydrogenase Inhibitor','alpha-Adrenergic Blocker','Antihistamine','Anxiolytics / Sedatives / Hypnotics'
        ,'Barbituates','Benzodiazepine','Benzodiazepines'
        ,'beta-Adrenergic Blocker','Beta-Blocker','Central alpha-2 Adrenergic Agonist'
        ,'gamma-Aminobutyric Acid A Receptor Agonist','gamma-Aminobutyric Acid A Receptor Positive Modulator'
        ,'General Anesthetic','Histamine-1 Receptor Antagonist','Influenza A M2 Protein Inhibitor'
        ,'Melatonin Receptor Agonist','Monoamine Oxidase Type B Inhibitor','Opioid Antagonist'
        ,'Orexin Receptor Antagonist','Partial Opioid Agonist','Sympathomimetic Amine Anorectic','Thiazide Diuretic'] 

    both = [
        'Aminoketone','Anti-epileptic Agent','Antipsychotics','Atypical Antipsychotic',
        'Central Nervous System Stimulant','Dopamine-2 Receptor Antagonist','Mood Stabilizer',
        'Mood Stabilizers','Norepinephrine Reuptake Inhibitor','Phenothiazine',
        'Stimulants','Sympathomimetic-like Agent','Tricyclic Antidepressant','Typical Antipsychotic']

    primary = [
        'Antidepressants','Monoamine Oxidase Inhibitor','Selective Serotonin Reuptake Inhibitor'
        'Serotonin and Norepinephrine Reuptake Inhibitor','Serotonin Reuptake Inhibitor','Substance Use Treatments']
    
    if med_class in augmentation: 
        return 'Augmentation'
    elif med_class in both: 
        return 'Both'
    elif med_class in primary: 
        return 'Primary'
    else: 
        return 'Unknown type'

#-----------------------List to string-----------------------------------------------
def list_to_value(list_val): 
    return ";;; ".join(map(str, list_val))

#-----------------------Text to disease-----------------------------------------------
'''
def text_to_disease(drug_name):
    print(drug_name)
    fda_url = f"https://api.fda.gov/drug/label.json?search=openfda.brand_name:\"{drug_name.lower()}\""
    fda_resp = requests.get(fda_url).json()
    results = fda_resp.get('results', [])

    if results and 'indications_and_usage' in results[0]:
        indication_large_body = results[0]['indications_and_usage'][0]
    else:
        return "No indications info found","No indication info found"

    #print(indication_large_body)

    nlp = spacy.load("en_ner_bc5cdr_md")
    nlp.get_pipe('ner')

    text_2 = ''.join(c.lower() if c.isalpha() else c for c in indication_large_body)

    result = nlp(text_2)
    result_2 = list(result.ents)
    result_3 = list(map(lambda x:str(x),result.ents))

    result_4 = [i for i in result_3 if i != drug_name]
    counted = Counter(result_4)  # Count occurrences
    sorted_result = tuple(sorted(counted.items(), key=lambda x: x[1], reverse=True))  # Sort by count (descending)    
    return indication_large_body,sorted_result
'''


nlp = spacy.load("en_ner_bc5cdr_md")

def text_to_disease(drug_name):
    #print(f"Processing: {drug_name}")
    
    # Construct FDA URL
    fda_url = f'https://api.fda.gov/drug/label.json?search=openfda.brand_name:"{drug_name.lower()}"'
    
    try:
        fda_resp = requests.get(fda_url).json()
        results = fda_resp.get('results', [])

        if results and 'indications_and_usage' in results[0]:
            indication_large_body = results[0]['indications_and_usage'][0]
        else:
            return "No indications info found", []
    except Exception as e:
        print(f"Error fetching FDA data for {drug_name}: {e}")
        return "No indications info found", []

    # Clean and process text
    text_clean = ''.join(c.lower() if c.isalpha() else c for c in indication_large_body)
    result = nlp(text_clean)
    result_ents = [str(ent) for ent in result.ents if str(ent).lower() != drug_name.lower()]

    # Count and sort
    counted = Counter(result_ents)
    sorted_result = sorted(counted.items(), key=lambda x: x[1], reverse=True)

    return indication_large_body, sorted_result

# Safe wrapper to ensure consistency
def safe_text_to_disease(drug_name):
    try:
        output = text_to_disease(drug_name)
        if isinstance(output, tuple) and len(output) == 2:
            return output
        else:
            return "No indications info found", []
    except Exception as e:
        print(f"Exception in processing {drug_name}: {e}")
        return "No indications info found", []

In [32]:
get_chembl_moa('sertraline')

[]

In [30]:
get_chembl_moa_from_rxcui(197536)

[]

In [13]:
df.columns

Index(['rxcui', 'medication_name', 'ingredient_name', 'ingredient_tty',
       'class_as_per_rxnav_rxcui'],
      dtype='object')

In [13]:
safe_text_to_disease('suvorexant')

('No indications info found', [])

In [13]:
# -------------------- Get Mechanism of Action from chembl by RXCUI & Medication_name - Taking 1 hr-----------------------
df['MOA_Version_1'] = df['rxcui'].apply(get_chembl_moa_from_rxcui)
df['MOA_Version_2'] = df.loc[df['MOA_Version_1'].apply(len) == 0,'medication_name'].apply(get_chembl_moa)

[Error] No drug name found for RxCUI 1013937
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float NaN to integer
[Error] Could not resolve drug name from RxCUI nan: cannot convert float Na

In [14]:
cond1 = df['MOA_Version_1'].fillna('').str.len() > 0
cond2 = df['MOA_Version_2'].fillna('').str.len() > 0

# Use np.select to assign values based on the conditions
df['MOA_Final'] = np.select(
    [cond1, cond2],
    [df['MOA_Version_1'], df['MOA_Version_2']],
    default=np.nan
)
df['MOA_Final'].fillna('', inplace=True)

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_18676/2950165165.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['MOA_Final'].fillna('', inplace=True)


In [15]:
#df

In [16]:
df['mechanism'] = df['MOA_Final'].apply(extract_mechanism)
df['target'] = df['MOA_Final'].apply(extract_target)

In [17]:
df_2=df

In [18]:
# Can be ignored for subsequent runs as this is a stop gap for the first run name error that was done. 
#df['MOA_Version_1'] = df['medication_name']
#df_2=df
#df_2['medication_name'] = df['medication_name']

In [19]:
df_2['drug_role'] = df_2['class_as_per_rxnav_rxcui'].apply(drug_role)

In [20]:
df_2.columns

Index(['rxcui', 'medication_name', 'ingredient_name', 'ingredient_tty',
       'class_as_per_rxnav_rxcui', 'MOA_Version_1', 'MOA_Version_2',
       'MOA_Final', 'mechanism', 'target', 'drug_role'],
      dtype='object')

In [21]:
df_2['clinical_drug_component'] = get_ntl_library_med(df_2.loc[:,['rxcui','medication_name']],"Clinical Drug Component")
df_2['RxNav_clinical_drug_component'] = df_2.loc[:,'clinical_drug_component'].apply(list_to_value)

[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.


In [22]:
df_2['Branded Drug Component'] = get_ntl_library_med(df_2.loc[:,['rxcui','medication_name']],"Branded Drug Component")

[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.
[Info] No RxCUI or drug name available.


In [23]:
df_2.rename(columns={'medication_name':'HL_Database_medication_name'
                     ,'mechanism':'mechanism_of_action'
                     ,'class_as_per_rxnav_rxcui':'Medication_class'},inplace=True)

In [24]:
df_3 = df_2[['rxcui','HL_Database_medication_name','RxNav_clinical_drug_component'
      ,'ingredient_name','ingredient_tty','Medication_class','mechanism_of_action','target'
      ,'drug_role','Branded Drug Component']]

In [25]:
df_3.columns

Index(['rxcui', 'HL_Database_medication_name', 'RxNav_clinical_drug_component',
       'ingredient_name', 'ingredient_tty', 'Medication_class',
       'mechanism_of_action', 'target', 'drug_role', 'Branded Drug Component'],
      dtype='object')

In [26]:
df_3.rename(columns={'Branded Drug Component':'generic_brand_drug_component'},inplace=True)
df_3['generic_drugs_exists'] = df_3['generic_brand_drug_component'].apply(lambda x: True if len(x)>0 else False)

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_18676/38922903.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3.rename(columns={'Branded Drug Component':'generic_brand_drug_component'},inplace=True)
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_18676/38922903.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3['generic_drugs_exists'] = df_3['generic_brand_drug_component'].apply(lambda x: True if len(x)>0 else False)


In [27]:
df_3.shape

(389, 11)

In [28]:
df_3.columns

Index(['rxcui', 'HL_Database_medication_name', 'RxNav_clinical_drug_component',
       'ingredient_name', 'ingredient_tty', 'Medication_class',
       'mechanism_of_action', 'target', 'drug_role',
       'generic_brand_drug_component', 'generic_drugs_exists'],
      dtype='object')

In [30]:
#df_3.tail(7)

In [217]:
#df_3.to_csv('Bio_Omics_Part1_V2.csv')

In [36]:
df_3_1 = df_3.loc[0:100]
df_3_2 = df_3.loc[101:200]
df_3_3 = df_3.loc[201:300]
df_3_4 = df_3.loc[301:]

In [37]:
df_3_1['Indications_V1'] = df_3_1['RxNav_clinical_drug_component'].apply(lambda x:get_chembl_indications_improved(x))

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_18676/2827124037.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3_1['Indications_V1'] = df_3_1['RxNav_clinical_drug_component'].apply(lambda x:get_chembl_indications_improved(x))


In [40]:
#df_3_1.head(5)

In [ ]:
#Run started at 01:34 PM - Ended at 2:00 PM

In [39]:
df_3_2['Indications_V1'] = df_3_2['RxNav_clinical_drug_component'].apply(lambda x:get_chembl_indications_improved(x))

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_18676/1549799011.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3_2['Indications_V1'] = df_3_2['RxNav_clinical_drug_component'].apply(lambda x:get_chembl_indications_improved(x))


In [ ]:
#Run started at 2:00 PM - Ended at 2.50 PM

In [41]:
df_3_3['Indications_V1'] = df_3_3['RxNav_clinical_drug_component'].apply(lambda x:get_chembl_indications_improved(x))

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_18676/102120114.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3_3['Indications_V1'] = df_3_3['RxNav_clinical_drug_component'].apply(lambda x:get_chembl_indications_improved(x))


In [ ]:
#Run started at 2:55 PM - Ended at 3.38 PM

In [49]:
df_3_4['Indications_V1'] = df_3_4['RxNav_clinical_drug_component'].apply(lambda x:get_chembl_indications_improved(x))
#Run started at 6:57 PM - 


prazosin 1 MG
prazosin 1 MG


prazosin 5 MG



pregabalin 165 MG;;; pregabalin 330 MG;;; pregabalin 82.5 MG;;; pregabalin 50 MG/ML;;; pregabalin 100 MG;;; pregabalin 150 MG;;; pregabalin 25 MG;;; pregabalin 300 MG;;; pregabalin 200 MG;;; pregabalin 50 MG;;; pregabalin 75 MG;;; pregabalin 225 MG;;; pregabalin 20 MG/ML
pregabalin 25 MG
propranolol hydrochloride 4.28 MG/ML;;; propranolol hydrochloride 120 MG;;; propranolol hydrochloride 160 MG;;; propranolol hydrochloride 80 MG;;; propranolol hydrochloride 40 MG;;; propranolol hydrochloride 1 MG/ML;;; propranolol hydrochloride 10 MG;;; propranolol hydrochloride 20 MG;;; propranolol hydrochloride 60 MG;;; propranolol hydrochloride 10 MG/ML;;; propranolol hydrochloride 16 MG/ML;;; propranolol hydrochloride 2 MG/ML;;; propranolol hydrochloride 4 MG/ML;;; propranolol hydrochloride 8 MG/ML;;; propranolol hydrochloride 80 MG/ML;;; propranolol hydrochloride 90 MG
propranolol hydrochloride 10 MG
propranolol hydrochloride 20 MG
quetiapine 100 MG


/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_18676/2544977129.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3_4['Indications_V1'] = df_3_4['RxNav_clinical_drug_component'].apply(lambda x:get_chembl_indications_improved(x))


In [51]:
df_4 = pd.concat([df_3_1, df_3_2, df_3_3 ,df_3_4], ignore_index=True)

In [53]:
#df_4.to_csv('Bio_Omics_Part1_V3.csv')

In [119]:
df_4[['HL_DB_name_indication', 'HL_DB_category']] = df_4['HL_Database_medication_name'].apply(lambda x: pd.Series(safe_text_to_disease(x)))

Processing: 1 ml diphenhydramine hydrochloride 50 mg/ml prefilled syringe
Processing: 1 ml haloperidol 5 mg/ml injection
Processing: 12 hr clonidine hydrochloride 0.1 mg extended release oral tablet
Processing: 12 hr clonidine hydrochloride 0.2 mg extended release oral tablet
Processing: 2 ml aripiprazole 200 mg/ml prefilled syringe
Processing: 24 hr bupropion hydrochloride 150 mg extended release oral tablet
Processing: 24 hr bupropion hydrochloride 150 mg extended release oral tablet [wellbutrin]
Processing: 24 hr bupropion hydrochloride 300 mg extended release oral tablet [wellbutrin]
Processing: 24 hr desvenlafaxine succinate 100 mg extended release oral tablet
Processing: 24 hr desvenlafaxine succinate 25 mg extended release oral tablet
Processing: 24 hr desvenlafaxine succinate 50 mg extended release oral tablet
Processing: 24 hr divalproex sodium 250 mg extended release oral tablet
Processing: 24 hr divalproex sodium 500 mg extended release oral tablet
Processing: 24 hr lamotrig

In [122]:
df_4[['Ing_name_indication', 'Ing_name_category']] = df_4['ingredient_name'].apply(lambda x: pd.Series(safe_text_to_disease(x)))

Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'floa

In [126]:
df_4.columns

Index(['rxcui', 'HL_Database_medication_name', 'RxNav_clinical_drug_component',
       'ingredient_name', 'ingredient_tty', 'Medication_class',
       'mechanism_of_action', 'target', 'drug_role',
       'generic_brand_drug_component', 'generic_drugs_exists',
       'Indications_V1', 'HL_DB_name_indication', 'HL_DB_category',
       'Ing_name_indication', 'Ing_name_category'],
      dtype='object')

In [140]:
# The entire usage & indication informationn though only coming from FDA. 
# This is a combination of HL DB stored name, if no information then this will search for Ingredient name. 
df_4['Indication_text_FDA'] = [None if x==y=='No indications info found' else y if x=='No indications info found' else x 
             for x,y in zip(df_4['HL_DB_name_indication'],df_4['Ing_name_indication'])]

In [148]:
# The terms of conditions extracted from the usage & indication text. 
# This is a combination of HL DB stored name, if no information then this will search for Ingredient name. 

df_4['Indication_terms_FDA'] = [None if x==y==[] else y if x==[] else x 
             for x,y in zip(df_4['HL_DB_category'],df_4['Ing_name_category'])]

In [150]:
df_4.columns

Index(['rxcui', 'HL_Database_medication_name', 'RxNav_clinical_drug_component',
       'ingredient_name', 'ingredient_tty', 'Medication_class',
       'mechanism_of_action', 'target', 'drug_role',
       'generic_brand_drug_component', 'generic_drugs_exists',
       'Indications_V1', 'HL_DB_name_indication', 'HL_DB_category',
       'Ing_name_indication', 'Ing_name_category',
       'Indication_text_chembl_FDA', 'Indication_terms_chembl_FDA',
       'Indication_terms_FDA', 'Indication_text_FDA'],
      dtype='object')

In [151]:
df_5 = df_4[['rxcui','HL_Database_medication_name', 'RxNav_clinical_drug_component'
          ,'ingredient_name', 'ingredient_tty', 'Medication_class'
          , 'mechanism_of_action', 'target', 'drug_role'
          , 'generic_brand_drug_component', 'generic_drugs_exists'
          ,  'Indications_V1', 'Indication_text_FDA', 'Indication_terms_FDA']]

In [152]:
df_5.shape

(389, 14)

In [154]:
df_5.to_csv('Bio\Omics_Part_1_V4.csv')

<>:1: SyntaxWarning: invalid escape sequence '\O'
<>:1: SyntaxWarning: invalid escape sequence '\O'
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_18676/2666942929.py:1: SyntaxWarning: invalid escape sequence '\O'
  df_5.to_csv('Bio\Omics_Part_1_V4.csv')


In [ ]:
df_3['Indications_V2'] = df_3['HL_Database_medication_name'].apply(lambda x:get_chembl_indications_improved(x))

In [196]:
ind_url = f"https://www.ebi.ac.uk/chembl/api/data/drug_indication.json?molecule_chembl_id=CHEMBL133572"
#print(ind_url)
ind_data = requests.get(ind_url).json()
print(ind_data)

{'drug_indications': [], 'page_meta': {'limit': 20, 'next': None, 'offset': 0, 'previous': None, 'total_count': 0}}


In [209]:
print(get_chembl_indications_improved('methylphenidate hydrochloride 36 MG'))

attention deficit hyperactivity disorder; chronic progressive multiple sclerosis; relapsing-remitting multiple sclerosis; conduct disorder; Fatigue; nicotine dependence; Sleep Disorder; depressive disorder; schizophrenia; childhood apraxia of speech; cancer; methamphetamine dependence; brain neoplasm; eating disorder; post-traumatic stress disorder; prostate carcinoma; post-operative sign or symptom; cocaine dependence; lymphoid neoplasm; drug dependence; orthostatic hypotension; opioid dependence; Dementia; delirium; major depressive disorder; prostate cancer; brain injury; idiopathic pulmonary fibrosis


In [219]:
test = ['diphenhydramine hydrochloride 50 MG/ML','haloperidol 5 MG/ML','hydroxyzine pamoate']

In [ ]:
def get_chembl_indications_improved(column):
    # Step 1: Get ChEMBL ID
    for i in column:
        search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
        res = requests.get(search_url).json()
    #print(len(res['molecules']))
    #print('-----------------------')
    #print(drug_name)
    try:
        #chembl_id = res["molecules"][0]["molecule_chembl_id"]
        chembl_ids = [m["molecule_chembl_id"] for m in res["molecules"] if "molecule_chembl_id" in m]
        my_set |= {"molecule_chembl_id"} for m in res["molecules"] if "molecule_chembl_id" not in my_set else set()
        
        #print(chembl_ids)
    except (KeyError, IndexError):
        #print(f"No ChEMBL ID found for {drug_name}")
        
    


In [225]:
def get_chembl_indications_improved_test(drug_list):
    unique_ids = set()
    for drug_name in drug_list:
        try:
            search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
            res = requests.get(search_url).json()
            chembl_ids = [m["molecule_chembl_id"] for m in res.get("molecules", []) if "molecule_chembl_id" in m]
            unique_ids.update(chembl_ids)
        except Exception as e:
            print(f"[Warning] Could not fetch ChEMBL ID for '{drug_name}': {e}")
            continue
    return unique_ids

In [228]:
all_chembl_id = get_chembl_indications_improved_test(df_3['HL_Database_medication_name'])

[Warning] Could not fetch ChEMBL ID for 'amphetamine aspartate': HTTPSConnectionPool(host='www.ebi.ac.uk', port=443): Read timed out. (read timeout=None)


In [230]:
#all_chembl_id

### DF_3 creation which has drug basic info MOA, Target & Generic_Drug_Exists related information only. Only the columns that are most important will be retained. 

In [92]:
df_2.columns
df_3 = df_2[['rxcui','medication_name','ingredient_name','ingredient_tty','med_first_name','med_class','drug_role','mechanism'
            ,'target','generic_list','generic_drugs_exists','Indications']]

In [94]:
df_3.to_csv('df_3.csv')

In [222]:
adverse_reactions_table = [
        "<table cellspacing=\"0\" cellpadding=\"0\" border=\"0\" width=\"100%\"><caption>Table 1: Adverse Reactions Occurring in &#x2265;1% in Alprazolam-treated Patients and Greater than Placebo-treated Patients in Placebo-Controlled Trials for Generalized Anxiety</caption><col width=\"36.96%\"/><col width=\"30.96%\"/><col width=\"32.08%\"/><tbody><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\">  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\"><content styleCode=\"bold\">Alprazolam</content> <content styleCode=\"bold\">n=565</content> </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\"><content styleCode=\"bold\">Placebo</content> <content styleCode=\"bold\">n=505</content> </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Nervous system disorders</content>  Drowsiness   Light-headedness   Dizziness   Akathisia  <content styleCode=\"bold\">Gastrointestinal disorders</content>  Dry mouth   Increased salivation  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   41%   21%   2%   2%     15%   4%  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   22%   19%   1%   1%     13%   2%  </td></tr><tr><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Cardiovascular disorders</content>  Hypotension  <content styleCode=\"bold\">Skin and subcutaneous tissue disorders</content>  Dermatitis/allergy  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   5%     4%  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   2%     3%  </td></tr></tbody></table>",
        "<table cellspacing=\"0\" cellpadding=\"0\" border=\"0\" width=\"100%\"><caption>Table 2: Adverse Reactions Occuring in &#x2265;1% in Alprazolam-treated Patients and Greater than Placebo-treated Patients in Placebo-Controlled Trials (Up to 10 Weeks) for Panic Disorder</caption><col width=\"36.92%\"/><col width=\"30.94%\"/><col width=\"32.14%\"/><tbody><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\"> </content> </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\"><content styleCode=\"bold\">Alprazolam</content> <content styleCode=\"bold\">n=1388</content> </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\"><content styleCode=\"bold\">Placebo</content> <content styleCode=\"bold\">n=1231</content> </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"> Drowsiness   Fatique and Tiredness   Impaired Coordination   Irritability   Memory Impairment   Cognitive Disorder   Decreased Libido   Dysartharia   Confusional state   Increased libido   Change in libido (not specified)   Disinhibition   Talkativeness   Derealization  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">77%   49%   40%   33%   33%   29%   14%   23%   10%   8%   7%   3%   2%   2%  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">43%   42%   18%   30%   22%   21%   8%   6%   8%   4%   6%   2%   1%   1%  </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"middle\"><content styleCode=\"bold\">Gastrointestinal disorders</content>  Constipation   Increased salivation  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   26%   6%  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   15%   4%  </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Skin and subcutaneous tissue disorders</content>  Rash  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   11%  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   8%  </td></tr><tr><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Other</content>  Increased appetite   Decreased appetite   Weight gain   Weight loss   Micturition difficulties   Menstrual disorders   Sexual dysfunction   Incontinence  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   33%   28%   27%   23%   12%   11%   7%   2%  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   23%   24%   18%   17%   9%   9%   4%   1%  </td></tr></tbody></table>",
        "<table cellspacing=\"0\" cellpadding=\"0\" border=\"0\" width=\"100%\"><caption>Table 3: Discontinuation-Emergent Symptom Incidence Reported in &#x2265;5% of Alprazolam-treated Patients and &gt; Placebo-treated Patients</caption><col width=\"50%\"/><col width=\"50%\"/><tfoot><tr><td colspan=\"2\">n=number of patients.</td></tr></tfoot><tbody><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\">  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\"><content styleCode=\"bold\">Alprazolam-treated Patients</content> <content styleCode=\"bold\">n=641</content> </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Nervous system disorders</content>  Insomnia   Light-headedness   Abnormal involuntary movement   Headache   Muscular twitching   Impaired coordination   Muscle tone disorders   Weakness  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   29.5%   19.3%   17.3%   17.0%   6.9%   6.6%   5.9%   5.8%  </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Psychiatric disorders</content>  Anxiety   Fatigue and Tiredness   Irritability   Cognitive disorder   Memory impairment   Depression   Confusional state  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   19.2%   18.4%   10.5%   10.3%   5.5%   5.1%   5.0%  </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Gastrointestinal disorders</content>  Nausea/Vomiting   Diarrhea   Decreased salivation  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   16.5%   13.6%   10.6%  </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Metabolism and nutrition disorders</content>  Weight loss   Decreased appetite  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   13.3%   12.8%  </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Dermatological disorders</content>  Sweating  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   14.4%  </td></tr><tr styleCode=\"Botrule\"><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Cardiovascular disorders</content>  Tachycardia  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   12.2%  </td></tr><tr><td align=\"justify\" styleCode=\"Lrule Rrule\" valign=\"top\"><content styleCode=\"bold\">Special Senses</content>  Blurred vision  </td><td align=\"center\" styleCode=\"Rrule\" valign=\"middle\">   10.0%  </td></tr></tbody></table>"
      ],

In [9]:
import requests
import pandas as pd
from io import StringIO

def fetch_adverse_reactions_html(drug_name):
    # Call FDA Drug Label API for the brand name
    url = f"https://api.fda.gov/drug/label.json?search=openfda.brand_name:\"{drug_name}\""
    response = requests.get(url)

    if response.status_code != 200:
        print("API Error:", response.status_code)
        return []

    data = response.json()

    # Extract the 'adverse_reactions_table' field
    results = data.get("results", [])
    if not results:
        print("No results found.")
        return []

    html_tables = results[0].get("adverse_reactions_table", [])
    return html_tables

def parse_adverse_reactions_tables(html_list):
    all_tables = []

    for html in html_list:
        try:
            html_io = StringIO(html)
            df_list = pd.read_html(html_io, flavor='lxml')
            all_tables.extend(df_list)
        except Exception as e:
            print("Error parsing table:", e)

    return all_tables

# 🚀 Example: Use the drug "Alprazolam"
html_tables = fetch_adverse_reactions_html("Alprazolam")
parsed_tables = parse_adverse_reactions_tables(html_tables)

# 💡 Show the first parsed table
if parsed_tables:
    print("\n--- First Adverse Reaction Table ---")
    print(parsed_tables[0])
else:
    print("No adverse reaction tables found.")



--- First Adverse Reaction Table ---
                                                   0                     1  \
0                                                NaN      Alprazolam n=565   
1  Nervous system disorders Drowsiness Light-head...  41% 21% 2% 2% 15% 4%   
2  Cardiovascular disorders Hypotension Skin and ...                 5% 4%   

                      2  
0         Placebo n=505  
1  22% 19% 1% 1% 13% 2%  
2                 2% 3%  


In [16]:
print(parsed_tables[2])

                                                   0  \
0                                                NaN   
1  Nervous system disorders Insomnia Light-headed...   
2  Psychiatric disorders Anxiety Fatigue and Tire...   
3  Gastrointestinal disorders Nausea/Vomiting Dia...   
4  Metabolism and nutrition disorders Weight loss...   
5                  Dermatological disorders Sweating   
6               Cardiovascular disorders Tachycardia   
7                      Special Senses Blurred vision   
8                              n=number of patients.   

                                             1  
0            Alprazolam-treated Patients n=641  
1  29.5% 19.3% 17.3% 17.0% 6.9% 6.6% 5.9% 5.8%  
2       19.2% 18.4% 10.5% 10.3% 5.5% 5.1% 5.0%  
3                            16.5% 13.6% 10.6%  
4                                  13.3% 12.8%  
5                                        14.4%  
6                                        12.2%  
7                                        10.0% 

('No indications info found', [])